# Kota Sentosa NM No-Show — Analysis, Word Report & PowerPoint Generator

This notebook is the full, runnable Python pipeline used to produce:
1. **`NM_NoShow_Project_Report.docx`** — the working analysis report
2. **`NM_NoShow_Presentation.pptx`** — the updated 18-content-slide deck (refreshed figures/numbers, plus the Declaration of Originality and Generative AI Declaration front-matter slides, plus the 12pt-minimum font fix)

It is organised in four parts:

| Part | What it does |
|---|---|
| **0. Setup** | installs/imports everything needed |
| **1. Analysis** | loads the xls, cleans/transforms it, trains Logistic Regression + Decision Tree, runs SHAP, computes segment and isotope-waste numbers, saves every figure |
| **2. Word report** | builds `NM_NoShow_Project_Report.docx` from scratch with `python-docx`, using the numbers/figures from Part 1 |
| **3. PowerPoint update** | takes your **existing** `NM_NoShow_Presentation.pptx` template, swaps in the refreshed figures/numbers, adds the two required declaration slides, and fixes any sub-12pt text |

## Inputs you need to upload in Colab
- `Kota_Sentosa_Hospital_Appointment_Data_NM.xls` — the source dataset
- `NM_NoShow_Presentation.pptx` — your **original/previous** deck (Part 3 edits this one in place; if you don't have a prior deck, skip Part 3 and just use the figures/metrics from Part 1)

Run the cells top to bottom. A file-upload prompt appears in Part 0.


## 0. Setup

In [ ]:
# Colab packages not preinstalled
!pip -q install xlrd shap python-docx python-pptx

import os, json, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import roc_auc_score, roc_curve, recall_score, precision_score, confusion_matrix

import shap

print("Setup complete.")

In [ ]:
# Upload inputs (Colab file picker).
# Expected: Kota_Sentosa_Hospital_Appointment_Data_NM.xls
#           NM_NoShow_Presentation.pptx   (your existing deck to update — optional, needed for Part 3 only)
from google.colab import files

print("Upload Kota_Sentosa_Hospital_Appointment_Data_NM.xls (and, if you have it, your existing NM_NoShow_Presentation.pptx):")
uploaded = files.upload()

XLS_PATH = next((f for f in uploaded if f.lower().endswith(".xls") or f.lower().endswith(".xlsx")), None)
PPTX_TEMPLATE_PATH = next((f for f in uploaded if f.lower().endswith(".pptx")), None)

print("Dataset file:", XLS_PATH)
print("Existing deck (optional, for Part 3):", PPTX_TEMPLATE_PATH)

os.makedirs("analysis", exist_ok=True)
OUT = "analysis"  # all generated figures/artifacts go here

## 1. Analysis Pipeline

Loads the appointment data, cleans/transforms predictors, trains two class-weighted
models, runs SHAP, and computes every number and figure used in the report and deck.
This mirrors the methodology described in the original project report:
log-transform the skewed booking lead time, group rare procedure categories,
consolidate inconsistent category labels, stratify the split, and class-weight
both models to handle the ~5% no-show rate.


In [ ]:
RANDOM_STATE = 42

NAVY = "#1F3557"
TEAL = "#1B8A8A"
CORAL = "#D9634C"
GOLD = "#E0A72E"
GREY = "#7C8896"
LIGHT_GREY = "#D8DEE6"

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "axes.edgecolor": "#B9C2CC",
    "axes.labelcolor": "#33404D",
    "text.color": "#22303D",
    "xtick.color": "#33404D",
    "ytick.color": "#33404D",
    "axes.grid": False,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
})

df = pd.read_excel(XLS_PATH, sheet_name="NM")
n_total = len(df)
n_raw_fields = df.shape[1]
print("Loaded:", df.shape)

In [ ]:
# --- clean / transform -------------------------------------------------------
work = df.copy()

# consolidate duplicate category spellings
for col in ["Houseownership", "Livinginstudycentercatchmentarera"]:
    work[col] = work[col].replace({"Not defined": "Not Defined"})

# rare-category grouping on Procedure Name
rare_procs = ["Parathyroid Scan", "Lung Scan - Perfusion Only"]
proc_counts_before = work["Procedure Name"].value_counts()
work["Procedure Name"] = work["Procedure Name"].replace({p: "Other NM Study" for p in rare_procs})

# booking lead time: log(1+x) transform
lead_raw_col = "ApptWaitingDaysDateofApptCreationApptDate"
lead_raw = work[lead_raw_col].astype(float)
skew_before = lead_raw.skew()
work["log_lead_time"] = np.log1p(lead_raw)
skew_after = work["log_lead_time"].skew()

# target
y = work["noshow_status"].astype(int)
n_noshow = int(y.sum())
n_attend = int((y == 0).sum())
noshow_rate = n_noshow / n_total * 100
imbalance_ratio = n_attend / n_noshow

sp_missing_pct = work["ServiceProvider"].isna().mean() * 100

cat_features = [
    "Ageyear", "Gender", "Ethnic", "SubsidyPaymentClass", "Houseownership",
    "Livinginstudycentercatchmentarera", "AppointmentDay", "AppointmentMonth",
    "Session", "Procedure Name",
]
num_features = ["AppointmentHour", "ApptSlotDurmins", "log_lead_time"]

X = work[cat_features + num_features].copy()

preprocessor = ColumnTransformer(transformers=[
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
    ("num", "passthrough", num_features),
])

print(f"Rows={n_total}  No-shows={n_noshow} ({noshow_rate:.2f}%)  Imbalance ratio={imbalance_ratio:.1f}:1")
print(f"Lead-time skew before={skew_before:.2f}  after log1p={skew_after:.2f}")

In [ ]:
# --- split & models ------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)
train_noshow_rate = y_train.mean() * 100
test_noshow_rate = y_test.mean() * 100

logreg = Pipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE)),
])
logreg.fit(X_train, y_train)

dtree = Pipeline([
    ("prep", preprocessor),
    ("clf", DecisionTreeClassifier(max_depth=5, min_samples_leaf=20,
                                    class_weight="balanced", random_state=RANDOM_STATE)),
])
dtree.fit(X_train, y_train)

def eval_model(pipe, X_test, y_test):
    proba = pipe.predict_proba(X_test)[:, 1]
    pred = pipe.predict(X_test)
    return {
        "roc_auc": roc_auc_score(y_test, proba),
        "recall": recall_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "proba": proba, "pred": pred,
    }

lr_eval = eval_model(logreg, X_test, y_test)
dt_eval = eval_model(dtree, X_test, y_test)
cm = confusion_matrix(y_test, dt_eval["pred"])

print("Logistic Regression:", {k: round(v, 3) for k, v in lr_eval.items() if k in ("roc_auc","recall","precision")})
print("Decision Tree      :", {k: round(v, 3) for k, v in dt_eval.items() if k in ("roc_auc","recall","precision")})

### Figures — lead-time transform, class imbalance, ROC, confusion matrix, decision tree

Each figure is saved twice: a report-friendly size (used in the Word doc) and a
size matched to the exact PowerPoint placeholder aspect ratio (`*_ppt.png`, used in
Part 3), so nothing gets stretched when it's dropped into the deck.


In [ ]:
# FIGURE 1 - lead time before/after log transform (report + ppt sizes)
def make_fig1(figsize, suffix):
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    axes[0].hist(lead_raw, bins=40, color=NAVY, alpha=0.85)
    axes[0].set_title(f"Before transform (skew={skew_before:.2f})", fontsize=11)
    axes[0].set_xlabel("Lead time (days)"); axes[0].set_ylabel("Appointments")
    axes[1].hist(work["log_lead_time"], bins=40, color=TEAL, alpha=0.85)
    axes[1].set_title(f"After log(1+x) (skew={skew_after:.2f})", fontsize=11)
    axes[1].set_xlabel("log(1 + lead time)")
    for ax in axes:
        ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.savefig(f"{OUT}/fig1_leadtime_transform{suffix}.png", dpi=200)
    plt.close()

make_fig1((9.5, 3.4), "")
make_fig1((9.5, 3.45), "_ppt")   # ratio 2.754, matches slide7 placeholder

In [ ]:
# FIGURE 2 - class imbalance (report only)
fig, ax = plt.subplots(figsize=(4.4, 3.6))
bars = ax.bar(["Attended", "No-show"], [n_attend, n_noshow], color=[NAVY, CORAL], width=0.55)
for b, val in zip(bars, [n_attend, n_noshow]):
    pct = val / n_total * 100
    ax.text(b.get_x() + b.get_width()/2, val + 60, f"{val:,}\n({pct:.1f}%)",
            ha="center", va="bottom", fontsize=10, fontweight="bold")
ax.set_ylabel("Appointments"); ax.set_ylim(0, n_attend * 1.18)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig(f"{OUT}/fig2_class_imbalance.png", dpi=200)
plt.close()

In [ ]:
# FIGURE 3 - ROC curve comparison (report + ppt sizes)
fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_eval["proba"])
fpr_dt, tpr_dt, _ = roc_curve(y_test, dt_eval["proba"])

def make_fig3(figsize, suffix, fontsize=8):
    fig, ax = plt.subplots(figsize=figsize)
    ax.plot(fpr_lr, tpr_lr, color=NAVY, lw=2.2, label=f"Logistic Regression (AUC={lr_eval['roc_auc']:.3f})")
    ax.plot(fpr_dt, tpr_dt, color=CORAL, lw=2.2, label=f"Decision Tree (AUC={dt_eval['roc_auc']:.3f})")
    ax.plot([0, 1], [0, 1], color=LIGHT_GREY, lw=1, linestyle="--")
    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
    ax.legend(fontsize=fontsize, loc="lower right", frameon=False)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.savefig(f"{OUT}/fig3_roc_comparison{suffix}.png", dpi=200)
    plt.close()

make_fig3((4.6, 4.0), "")
make_fig3((4.6, 3.83), "_ppt", fontsize=8.5)   # ratio 1.201, matches slide10 box1

In [ ]:
# FIGURE 4 - confusion matrix (report + ppt sizes)
def make_fig4(figsize, suffix, fontsize=13):
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(cm, cmap="Blues")
    labels = ["Attended", "No-show"]
    ax.set_xticks([0, 1]); ax.set_xticklabels(labels)
    ax.set_yticks([0, 1]); ax.set_yticklabels(labels)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    for i in range(2):
        for j in range(2):
            color = "white" if cm[i, j] > cm.max()/2 else "#1a1a1a"
            ax.text(j, i, f"{cm[i,j]:,}", ha="center", va="center",
                     color=color, fontsize=fontsize, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{OUT}/fig4_confusion_matrix{suffix}.png", dpi=200)
    plt.close()

make_fig4((4.2, 4.0), "")
make_fig4((4.4, 4.0), "_ppt", fontsize=15)   # ratio 1.10, matches slide10 box2

In [ ]:
# FIGURE 5 - decision tree, top 3 levels (report + ppt sizes)
ohe = dtree.named_steps["prep"].named_transformers_["cat"]
feature_names = list(ohe.get_feature_names_out(cat_features)) + num_features

def make_fig5(figsize, suffix, fontsize=8):
    fig, ax = plt.subplots(figsize=figsize)
    plot_tree(dtree.named_steps["clf"], max_depth=3, feature_names=feature_names,
              class_names=["Attended", "No-show"], filled=True, rounded=True,
              fontsize=fontsize, ax=ax, impurity=False)
    plt.tight_layout()
    plt.savefig(f"{OUT}/fig5_decision_tree{suffix}.png", dpi=180)
    plt.close()

make_fig5((15, 7), "")
make_fig5((16, 4.75), "_ppt", fontsize=9)   # ratio 3.369, matches slide11 box

### SHAP analysis

Explains the Decision Tree's predictions on the held-out test set. `pretty_feature`
turns raw one-hot column names (e.g. `Procedure Name_Bone Scan`) into readable
labels (`Procedure Name: Bone Scan`) for the charts.


In [ ]:
X_test_enc = dtree.named_steps["prep"].transform(X_test)
if hasattr(X_test_enc, "toarray"):
    X_test_enc = X_test_enc.toarray()
X_test_enc_df = pd.DataFrame(X_test_enc, columns=feature_names)

explainer = shap.TreeExplainer(dtree.named_steps["clf"])
sv = explainer.shap_values(X_test_enc_df)
if isinstance(sv, list):
    shap_values = sv[1]
elif sv.ndim == 3:
    shap_values = sv[:, :, 1]
else:
    shap_values = sv

mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_rank = pd.Series(mean_abs_shap, index=feature_names).sort_values(ascending=False)

def pretty_feature(name):
    for raw in cat_features:
        prefix = raw + "_"
        if name.startswith(prefix):
            return f"{raw}: {name[len(prefix):]}"
    mapping = {"AppointmentHour": "Appointment hour", "ApptSlotDurmins": "Slot duration (mins)",
               "log_lead_time": "Booking lead time (log)"}
    return mapping.get(name, name)

top10 = shap_rank.head(10)
print(top10.rename(index=pretty_feature))

In [ ]:
# FIGURE 6 - SHAP summary beeswarm (report + ppt sizes)
def make_fig6(figsize, suffix):
    plt.figure(figsize=figsize)
    shap.summary_plot(shap_values, X_test_enc_df,
                       feature_names=[pretty_feature(f) for f in feature_names],
                       show=False, max_display=10, plot_size=None)
    plt.tight_layout()
    plt.savefig(f"{OUT}/fig6_shap_summary{suffix}.png", dpi=200, bbox_inches="tight")
    plt.close()

make_fig6((6.2, 4.4), "")
make_fig6((6.6, 5.15), "_ppt")   # ratio 1.282, matches slide12 box

In [ ]:
# FIGURE 7 - top 10 features, mean |SHAP| bar chart (report only)
fig, ax = plt.subplots(figsize=(6.2, 4.4))
order = top10.iloc[::-1]
ax.barh([pretty_feature(f) for f in order.index], order.values, color=TEAL)
ax.set_xlabel("Mean |SHAP value|")
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig(f"{OUT}/fig7_shap_top10.png", dpi=200)
plt.close()

### Segment analysis and isotope-waste estimate

In [ ]:
# no-show rate by procedure
by_proc = work.groupby("Procedure Name")["noshow_status"].agg(["mean", "count"])
by_proc["rate_pct"] = by_proc["mean"] * 100
by_proc = by_proc.sort_values("rate_pct", ascending=False)

# no-show rate by booking lead-time band
bins = [-1, 7, 30, 90, 180, 10000]
band_labels = ["0-7 days", "8-30 days", "31-90 days", "91-180 days", "181+ days"]
work["lead_band"] = pd.cut(lead_raw, bins=bins, labels=band_labels)
by_band = work.groupby("lead_band", observed=True)["noshow_status"].agg(["mean", "count"])
by_band["rate_pct"] = by_band["mean"] * 100
by_band = by_band.reindex(band_labels)

print(by_proc.round(2)); print(); print(by_band.round(2))

In [ ]:
# FIGURE 8 - segment rates (report + ppt sizes)
def make_fig8(figsize, suffix, fontsize=11):
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    axes[0].barh(by_proc.index[::-1], by_proc["rate_pct"][::-1], color=NAVY)
    axes[0].set_xlabel("No-show rate (%)"); axes[0].set_title("By procedure", fontsize=fontsize)
    axes[0].spines["top"].set_visible(False); axes[0].spines["right"].set_visible(False)
    axes[1].bar(by_band.index, by_band["rate_pct"], color=GOLD)
    axes[1].set_ylabel("No-show rate (%)"); axes[1].set_title("By booking lead-time band", fontsize=fontsize)
    axes[1].tick_params(axis="x", rotation=20)
    axes[1].spines["top"].set_visible(False); axes[1].spines["right"].set_visible(False)
    plt.tight_layout()
    plt.savefig(f"{OUT}/fig8_segment_rates{suffix}.png", dpi=200)
    plt.close()

make_fig8((10.2, 3.6), "")
make_fig8((15.3, 3.6), "_ppt")   # ratio 4.257, matches slide13 (wide banner)

In [ ]:
# isotope cost assumptions (illustrative NM planning figures — replace with actual procurement costs)
isotope_cost = {
    "Myocardial Perfusion Imaging": 450,   # Tc-99m Sestamibi/Tetrofosmin
    "Bone Scan": 280,                       # Tc-99m MDP
    "Renogram Study": 320,                  # Tc-99m MAG3
    "Thyroid Scan and Ultrasound": 180,     # Tc-99m Pertechnetate
    "Other NM Study": 300,                  # Parathyroid / Lung Scan blend
}
work["isotope_cost"] = work["Procedure Name"].map(isotope_cost)
noshow_rows = work[work["noshow_status"] == 1]
total_waste = float(noshow_rows["isotope_cost"].sum())
avg_slot_min = float(noshow_rows["ApptSlotDurmins"].mean())

waste_by_proc = noshow_rows.groupby("Procedure Name").apply(
    lambda g: pd.Series({"no_shows": len(g), "unit_cost": isotope_cost[g.name],
                          "total_waste": len(g) * isotope_cost[g.name]}),
    include_groups=False,
).sort_values("total_waste", ascending=False)

print(f"Total isotope waste: SGD {total_waste:,.0f}   Avg slot tied up: {avg_slot_min:.1f} min")
print(waste_by_proc)

In [ ]:
# FIGURE 9 - isotope waste by procedure (report + ppt sizes)
def make_fig9(figsize, suffix, fontsize=8):
    fig, ax = plt.subplots(figsize=figsize)
    bars = ax.bar(waste_by_proc.index, waste_by_proc["total_waste"], color=CORAL)
    ax.set_ylabel("Estimated wasted isotope cost (SGD)")
    ax.tick_params(axis="x", rotation=20)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    for b, val in zip(bars, waste_by_proc["total_waste"]):
        ax.text(b.get_x() + b.get_width()/2, val, f"{val:,.0f}", ha="center", va="bottom", fontsize=fontsize)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.savefig(f"{OUT}/fig9_isotope_waste{suffix}.png", dpi=200)
    plt.close()

make_fig9((5.4, 3.6), "")
make_fig9((7.3, 4.55), "_ppt", fontsize=9)   # ratio 1.604, matches slide14 box

In [ ]:
# --- collect every number used downstream into one dict -----------------
metrics = {
    "n_total": n_total, "n_raw_fields": n_raw_fields,
    "n_procedures": int(df["Procedure Name"].nunique()),
    "n_noshow": n_noshow, "n_attend": n_attend,
    "noshow_rate_pct": round(noshow_rate, 2), "imbalance_ratio": round(imbalance_ratio, 1),
    "sp_missing_pct": round(sp_missing_pct, 1),
    "skew_before": round(skew_before, 2), "skew_after": round(skew_after, 2),
    "lead_time_max_days": int(lead_raw.max()),
    "rare_proc_counts": {p: int(proc_counts_before[p]) for p in rare_procs},
    "train_rows": int(len(X_train)), "test_rows": int(len(X_test)),
    "train_noshow_rate_pct": round(train_noshow_rate, 2), "test_noshow_rate_pct": round(test_noshow_rate, 2),
    "logreg": {k: round(lr_eval[k], 3) for k in ("roc_auc","recall","precision")},
    "dtree": {k: round(dt_eval[k], 3) for k in ("roc_auc","recall","precision")},
    "confusion_matrix": cm.tolist(),
    "shap_top10": [{"feature": pretty_feature(f), "mean_abs_shap": round(float(v), 4)} for f, v in top10.items()],
    "by_procedure_rate": {i: {"rate_pct": round(r["rate_pct"], 2), "n": int(r["count"])} for i, r in by_proc.iterrows()},
    "by_lead_band_rate": {i: {"rate_pct": round(r["rate_pct"], 2), "n": int(r["count"])} for i, r in by_band.iterrows()},
    "isotope_cost_map": isotope_cost,
    "total_isotope_waste_sgd": round(total_waste, 0),
    "avg_slot_minutes_wasted": round(avg_slot_min, 1),
    "waste_by_procedure": {i: {"no_shows": int(r["no_shows"]), "unit_cost": int(r["unit_cost"]),
                                 "total_waste": round(float(r["total_waste"]), 0)} for i, r in waste_by_proc.iterrows()},
}
with open(f"{OUT}/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print(json.dumps(metrics, indent=2))

## 2. Build the Word Report (`python-docx`)

Builds `NM_NoShow_Project_Report.docx` end-to-end from the `metrics` dict and the
figures saved in Part 1. This is a **working document**, not the graded deliverable
(the module only grades the PowerPoint) — the note box on the cover page says so
explicitly.


In [ ]:
from docx import Document
from docx.shared import Inches, Pt, RGBColor, Cm
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.oxml.ns import qn
from docx.oxml import OxmlElement

NAVY_RGB = RGBColor(0x1F, 0x35, 0x57)
TEAL_RGB = RGBColor(0x1B, 0x8A, 0x8A)

def set_cell_shading(cell, hex_color):
    tcPr = cell._tc.get_or_add_tcPr()
    shd = OxmlElement("w:shd")
    shd.set(qn("w:val"), "clear")
    shd.set(qn("w:fill"), hex_color)
    tcPr.append(shd)

def h1(doc, text):
    p = doc.add_heading(text, level=1)
    for run in p.runs:
        run.font.color.rgb = NAVY_RGB

def h2(doc, text):
    p = doc.add_heading(text, level=2)
    for run in p.runs:
        run.font.color.rgb = NAVY_RGB

def h3(doc, text):
    p = doc.add_heading(text, level=3)
    for run in p.runs:
        run.font.color.rgb = TEAL_RGB

def para(doc, text, italic=False, bold=False):
    p = doc.add_paragraph()
    r = p.add_run(text)
    r.italic = italic
    r.bold = bold
    return p

def bullet(doc, text):
    doc.add_paragraph(text, style="List Bullet")

def note_box(doc, title, lines):
    table = doc.add_table(rows=1, cols=1)
    table.alignment = WD_TABLE_ALIGNMENT.CENTER
    cell = table.rows[0].cells[0]
    set_cell_shading(cell, "EEF1F4")
    cell.paragraphs[0].add_run(title).bold = True
    for line in lines:
        p = cell.add_paragraph(line)
    doc.add_paragraph("")

def add_image(doc, path, width_in):
    p = doc.add_paragraph()
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    run = p.add_run()
    run.add_picture(path, width=Inches(width_in))

def caption(doc, text):
    p = doc.add_paragraph()
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    r = p.add_run(text)
    r.italic = True
    r.font.size = Pt(10)

def metric_table(doc, headers, rows):
    table = doc.add_table(rows=1, cols=len(headers))
    table.style = "Light Grid Accent 1"
    hdr = table.rows[0].cells
    for i, htext in enumerate(headers):
        hdr[i].text = str(htext)
        set_cell_shading(hdr[i], "1F3557")
        for r in hdr[i].paragraphs[0].runs:
            r.font.color.rgb = RGBColor(0xFF, 0xFF, 0xFF)
            r.bold = True
    for row in rows:
        cells = table.add_row().cells
        for i, val in enumerate(row):
            cells[i].text = str(val)
    doc.add_paragraph("")
    return table

print("Word-doc helper functions ready.")

In [ ]:
fmt = lambda n: f"{n:,}"
waste_fmt = f"SGD {fmt(int(metrics['total_isotope_waste_sgd']))}"
rareA = metrics["rare_proc_counts"]["Parathyroid Scan"]
rareB = metrics["rare_proc_counts"]["Lung Scan - Perfusion Only"]
byProc = metrics["by_procedure_rate"]
byBand = metrics["by_lead_band_rate"]

doc = Document()
for style_name in ("Normal",):
    style = doc.styles[style_name]
    style.font.name = "Calibri"
    style.font.size = Pt(11)

# ---------------- Title page ----------------
para(doc, "School of Informatics & IT", bold=True)
para(doc, "SPECIALIST DIPLOMA IN AI-DRIVEN DATA ANALYTICS (SDADDA)", bold=True)
para(doc, "AI-Driven Analytics with Large Language Models (CBA1C03)", bold=True)
doc.add_paragraph("")
p = doc.add_paragraph(); r = p.add_run("Reducing No-Show Related Waste in Nuclear Medicine Appointments:"); r.bold = True; r.font.size = Pt(15)
p = doc.add_paragraph(); r = p.add_run("A Predictive Analytics Approach for Kota Sentosa Hospital"); r.bold = True; r.font.size = Pt(15)
para(doc, "Project Analysis Report (Working Document) — generated end-to-end by this notebook", italic=True)

note_box(doc, "Note on this document", [
    "This report is a working analysis document prepared ahead of the required PowerPoint submission. "
    "Every figure, table, and headline number below was computed by this notebook from the source xls, "
    "not copied from a prior draft. The official CBA1C03 project brief requires an 18-slide deck "
    "(Design > Slide Size > Widescreen) — that PowerPoint, not this document, is the graded deliverable. "
    "Use this report as source material for the deck.",
])
doc.add_page_break()

# ==================== 1. INTRODUCTION AND CONTEXT ====================
h1(doc, "1. Introduction and Context")
h2(doc, "1a. Generative AI First Draft")
para(doc, 'Prompt used: "Draft a business problem statement, persona, and Job-To-Be-Done framing for '
          'reducing patient no-shows in a hospital Nuclear Medicine department, considering that no-shows '
          'waste both appointment slots and pre-prepared radioactive isotopes."', italic=True)

note_box(doc, "AI Draft Summary", [
    "The AI proposed: no-shows are costly because appointment slots and staff time go unused; a persona of "
    "a 'Department Manager' who wants to reduce wasted resources; and a JTBD framed as 'predict which "
    "patients will not show up so staff can intervene.' It listed generic predictors such as age, "
    "appointment day, and distance from home.",
])

h3(doc, "Critique of the AI Draft")
bullet(doc, "The AI draft treated the problem generically, as if it were any outpatient clinic — it did not "
            "recognise that Nuclear Medicine (NM) procedures use radiopharmaceutical isotopes prepared or "
            "dosed specifically for a booked patient, which decay quickly and cannot simply be reused. This "
            "is the single most important domain fact and the AI missed it entirely.")
bullet(doc, 'The persona was too shallow ("wants to reduce waste") without concrete goals, daily behaviours, '
            "or how the persona would actually use a model's output.")
bullet(doc, "The suggested predictors were generic and not grounded in what is actually available in the "
            "Kota Sentosa dataset (e.g. it did not consider procedure type, lead time, or subsidy class).")
bullet(doc, "It did not address the severe class imbalance a real no-show dataset would have, which "
            "materially changes how the modelling should be approached.")

h3(doc, "Refined and Improved Version")
h3(doc, "Business Problem")
para(doc, f"Kota Sentosa Hospital's Nuclear Medicine (NM) department performs scans such as Myocardial "
          f"Perfusion Imaging (MPI), Bone Scans, and Renogram studies that rely on radiopharmaceutical "
          f"isotopes (e.g. Tc-99m based tracers). When a patient does not show up (\"no-show\"), the "
          f"department loses the appointment slot AND the isotope dose already prepared for that patient. "
          f"In this dataset, no-shows account for {metrics['noshow_rate_pct']}% of {fmt(metrics['n_total'])} "
          f"appointments ({fmt(metrics['n_noshow'])} cases), with an estimated wasted isotope cost of "
          f"approximately {waste_fmt} (see Section 3.3).")

h3(doc, "Key Beneficiary / Persona")
metric_table(doc, ["Persona attribute", "Description"], [
    ["Name / Role", "Priya, Nuclear Medicine Scheduling Coordinator"],
    ["Goals", "Maximise scanner utilisation; minimise wasted isotope orders"],
    ["Pain points", "Only finds out about a no-show after the isotope is already prepared, with no early warning"],
    ["Behaviours", 'Manually calls a few "high-risk-feeling" patients by gut feel, with no systematic prioritisation'],
    ["How she would use the solution", "Each morning, receives a flagged list ranked by predicted no-show risk to prioritise calls / delay isotope prep / activate standby patients"],
])

h3(doc, "Job-To-Be-Done (JTBD)")
note_box(doc, "JTBD Statement", [
    "When I am scheduling and preparing for tomorrow's Nuclear Medicine appointments, I want to know in "
    "advance which bookings are at high risk of a no-show, so I can intervene before the isotope is "
    "wasted and the slot is lost.",
])
para(doc, "Target variable: NoShowStatus / noshow_status (binary: 1 = did not attend, 0 = attended).")
para(doc, "Likely predictors: procedure type, booking lead time, appointment day/session/hour, age band, "
          "subsidy payment class, housing type, catchment residency.")
doc.add_page_break()

print("Section 1 written.")

In [ ]:
# ==================== 2. MODELLING APPROACH ====================
h1(doc, "2. Highlight of the Modelling Approach")
h2(doc, "2.1 Transforming Predictors to Uncover Useful Patterns")
para(doc, f"Booking lead time was strongly right-skewed (skewness = {metrics['skew_before']}); a log(1+x) "
          f"transform reduced this to {metrics['skew_after']}, letting the model learn from gradual "
          f"differences rather than being dominated by the small number of very-long-lead-time bookings "
          f"(max {fmt(metrics['lead_time_max_days'])} days).")
add_image(doc, f"{OUT}/fig1_leadtime_transform.png", 6.2)
caption(doc, "Figure 1. Booking lead time before and after log transformation.")

para(doc, f'Rare category grouping: Procedure Name had two categories with very few cases (Parathyroid '
          f'Scan, n={rareA}; Lung Scan – Perfusion Only, n={rareB}). These were grouped into a single '
          f'"Other NM Study" category so the model is not forced to learn unstable patterns from a '
          f'handful of examples.')
para(doc, 'Inconsistent category labels (e.g. "Not defined" vs "Not Defined") were consolidated to '
          "prevent the model from treating identical categories as distinct levels.")

h2(doc, "2.2 Handling Class Imbalance")
para(doc, f"No-shows are a small minority: only {metrics['noshow_rate_pct']}% of "
          f"{fmt(metrics['n_total'])} appointments ({fmt(metrics['n_noshow'])} cases), an imbalance ratio "
          f"of roughly {metrics['imbalance_ratio']}:1.")
add_image(doc, f"{OUT}/fig2_class_imbalance.png", 3.4)
caption(doc, "Figure 2. Class imbalance between attended and no-show appointments.")
bullet(doc, f"Stratified 75/25 train/test split preserves the ~{metrics['noshow_rate_pct']}% no-show "
            f"ratio in both sets (train {metrics['train_noshow_rate_pct']}%, test {metrics['test_noshow_rate_pct']}%).")
bullet(doc, "class_weight='balanced' was applied in both models so the rare no-show class is not ignored.")
bullet(doc, "Evaluation focus shifted from accuracy (misleading here) to recall and ROC-AUC for the no-show class.")

h2(doc, "2.3 Modelling Choices Made")
bullet(doc, "Baseline: Logistic Regression (class-weighted) — interpretable, tests for a roughly linear/additive relationship.")
bullet(doc, "Decision Tree (max_depth=5, min_samples_leaf=20, class-weighted) — produces simple if-then "
            "rules that can be shown directly to scheduling staff.")
add_image(doc, f"{OUT}/fig3_roc_comparison.png", 3.4)
caption(doc, "Figure 3. ROC curve comparison between the two models.")

metric_table(doc, ["Metric", "Logistic Regression", "Decision Tree (depth=5)"], [
    ["ROC-AUC", metrics["logreg"]["roc_auc"], metrics["dtree"]["roc_auc"]],
    ["Recall (No-show)", metrics["logreg"]["recall"], metrics["dtree"]["recall"]],
    ["Precision (No-show)", metrics["logreg"]["precision"], metrics["dtree"]["precision"]],
])

para(doc, "The Decision Tree achieves higher recall at a similar precision trade-off and produces "
          "directly explainable rules, so it is recommended for deployment; Logistic Regression is "
          "retained as a sanity-check baseline.")
add_image(doc, f"{OUT}/fig4_confusion_matrix.png", 3.3)
caption(doc, "Figure 4. Confusion matrix for the Decision Tree on the held-out test set.")
add_image(doc, f"{OUT}/fig5_decision_tree.png", 6.5)
caption(doc, "Figure 5. Top decision rules from the Decision Tree (first 3 levels).")
doc.add_page_break()

print("Section 2 written.")

In [ ]:
# ==================== 3. INSIGHTS AND JUSTIFICATIONS ====================
h1(doc, "3. Insights and Justifications")
h2(doc, "3.1 Main Model Insights (SHAP Analysis)")
para(doc, "SHAP values were computed on the Decision Tree against the held-out test set to identify which "
          "features most influence the model's no-show predictions.")
add_image(doc, f"{OUT}/fig6_shap_summary.png", 5.4)
caption(doc, "Figure 6. SHAP summary plot — feature impact on no-show risk.")
add_image(doc, f"{OUT}/fig7_shap_top10.png", 5.4)
caption(doc, "Figure 7. Top 10 features ranked by mean absolute SHAP value.")

for line in [
    "Procedure type (particularly Bone Scan vs. MPI) is the strongest signal.",
    "Ethnicity showed a measurable SHAP contribution — flagged as a fairness consideration, not an "
    "operational trigger for individual patients.",
    "Catchment-area residency contributes meaningfully, plausibly linked to travel burden.",
    "Booking lead time (log-transformed) is a consistent driver of risk.",
]:
    bullet(doc, line)

h2(doc, "3.2 No-show Patterns by Segment")
add_image(doc, f"{OUT}/fig8_segment_rates.png", 6.4)
caption(doc, "Figure 8. No-show rate by procedure and by booking lead-time band.")
para(doc, f"No-show rates rise from {byBand['0-7 days']['rate_pct']}% (booked within a week) to "
          f"{byBand['31-90 days']['rate_pct']}%–{byBand['91-180 days']['rate_pct']}% for bookings 31–180 "
          f"days out. Procedure-level differences are also visible: 'Other NM Study' "
          f"({byProc['Other NM Study']['rate_pct']}%) and MPI ({byProc['Myocardial Perfusion Imaging']['rate_pct']}%) "
          f"show the highest rates, well above Bone Scan ({byProc['Bone Scan']['rate_pct']}%).")

h2(doc, "3.3 Isotope and Slot Waste Estimate")
para(doc, f"Across the {fmt(metrics['n_noshow'])} no-shows, an estimated {waste_fmt} of prepared isotope "
          f"was wasted, with the average no-show tying up a scanner slot of roughly "
          f"{metrics['avg_slot_minutes_wasted']} minutes.")
add_image(doc, f"{OUT}/fig9_isotope_waste.png", 5.0)
caption(doc, "Figure 9. Estimated isotope waste from no-shows, by procedure.")
note_box(doc, "Isotope cost note", [
    "Per-procedure isotope costs used here are illustrative NM department planning estimates, not "
    "confirmed Kota Sentosa procurement prices. Replace with actual departmental costings before using "
    "these figures for budget decisions.",
])

h2(doc, "3.4 Main Business Actions and Implications")
for line in [
    "Tiered intervention by risk score: prioritise reminder calls for high-risk bookings; delay isotope "
    "prep for the highest-risk, highest-cost slots.",
    "Standby list activation for high-risk slots identified a few days out.",
    "Booking policy review: consider an interim reconfirmation call for very-long-lead-time bookings.",
    "Fairness safeguard: trigger interventions from operational signals (lead time, procedure), not "
    "demographic attributes; review with clinical governance before deployment.",
]:
    bullet(doc, line)

h2(doc, "3.5 Prepared Scenario / Use Case")
note_box(doc, "Scenario", [
    "A patient books an MPI scan 20 weeks in advance, living outside the hospital's catchment area. The "
    "model flags this booking as high-risk three days before the appointment. The coordinator calls to "
    "reconfirm; the patient can no longer make it. Because the flag came early, the slot is offered to a "
    "standby patient and the isotope order is adjusted before final preparation — avoiding both the "
    "wasted isotope cost and the empty scanner slot.",
])

h2(doc, "3.6 Limitations and Deployment Considerations")
for line in [
    f"Precision for the no-show class is low (~{round(metrics['dtree']['precision']*100)}%), so this model "
    f"should support human decision-making, not fully automate cancellations.",
    f"ServiceProvider was excluded due to {metrics['sp_missing_pct']}% missing values.",
    "Isotope costs in Section 3.3 are illustrative and should be validated against actual procurement figures.",
    "The ethnicity signal requires governance review before any operational use.",
    "Model should be retrained periodically as scheduling patterns or policy shift over time.",
]:
    bullet(doc, line)

doc.save("NM_NoShow_Project_Report.docx")
print("Saved NM_NoShow_Project_Report.docx")

## 3. Update the PowerPoint Deck

This part takes your **existing** `NM_NoShow_Presentation.pptx` (uploaded in Part 0)
and:
1. swaps in the refreshed `*_ppt.png` figures from Part 1 (matched by aspect ratio
   to each slide's image placeholder, so nothing stretches),
2. updates the handful of number callouts that changed,
3. bumps every text run under 12pt up to 12pt (CBA1C03 font-size requirement),
4. adds the required **Declaration of Originality** and **Generative AI
   Declaration** front-matter slides (these do not count toward the 18-slide
   content limit),
5. re-labels the agenda slide as the Table of Contents and tidies the references
   slide.

If you don't have a prior deck to update, skip this part — Parts 1–2 already give
you every figure and number you need to build slides from scratch.


In [ ]:
import re, glob, shutil, zipfile
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.dml.color import RGBColor
from pptx.enum.shapes import MSO_SHAPE

assert PPTX_TEMPLATE_PATH, "No .pptx was uploaded in Part 0 — skip Part 3 or re-run Part 0 with a deck attached."

WORKDIR = "pptx_work"
if os.path.exists(WORKDIR):
    shutil.rmtree(WORKDIR)
os.makedirs(WORKDIR)

with zipfile.ZipFile(PPTX_TEMPLATE_PATH) as z:
    z.extractall(WORKDIR)

print("Unpacked template to", WORKDIR)

In [ ]:
# --- 3.1 swap in the refreshed figures --------------------------------------
# Mapping is by content, established by inspecting each slide's image placeholder:
#   slide7  -> lead-time transform      slide11 -> decision tree
#   slide10 -> ROC curve + confusion    slide12 -> SHAP summary
#   slide13 -> segment rates            slide14 -> isotope waste
image_map = {
    "image1.png": f"{OUT}/fig1_leadtime_transform_ppt.png",
    "image2.png": f"{OUT}/fig3_roc_comparison_ppt.png",
    "image3.png": f"{OUT}/fig4_confusion_matrix_ppt.png",
    "image4.png": f"{OUT}/fig5_decision_tree_ppt.png",
    "image5.png": f"{OUT}/fig6_shap_summary_ppt.png",
    "image6.png": f"{OUT}/fig8_segment_rates_ppt.png",
    "image7.png": f"{OUT}/fig9_isotope_waste_ppt.png",
}
media_dir = os.path.join(WORKDIR, "ppt", "media")
for target, src in image_map.items():
    dest = os.path.join(media_dir, target)
    if os.path.exists(dest) and os.path.exists(src):
        shutil.copy(src, dest)
        print(f"Replaced {target}  <-  {src}")
    else:
        print(f"SKIPPED {target} (not found in this deck or figure missing)")

In [ ]:
# --- 3.2 update number callouts on the slides --------------------------------
# Each entry: (slide filename, old text, new text). Only applies if the old text
# is actually found (so this is safe to re-run against a similar but not
# identical deck without crashing).
text_updates = [
    ("slide4.xml",  "SGD 154K",       f"SGD {round(metrics['total_isotope_waste_sgd']/1000)}K"),
    ("slide9.xml",  "0.718",          f"{metrics['logreg']['roc_auc']:.3f}"),
    ("slide9.xml",  "0.101",          f"{metrics['logreg']['precision']:.3f}"),
    ("slide14.xml", "SGD 153,840",    f"SGD {int(metrics['total_isotope_waste_sgd']):,}"),
]

slides_dir = os.path.join(WORKDIR, "ppt", "slides")
for fname, old, new in text_updates:
    fpath = os.path.join(slides_dir, fname)
    if not os.path.exists(fpath):
        continue
    txt = open(fpath, encoding="utf-8").read()
    if f"<a:t>{old}</a:t>" in txt:
        txt = txt.replace(f"<a:t>{old}</a:t>", f"<a:t>{new}</a:t>")
        open(fpath, "w", encoding="utf-8").write(txt)
        print(f"{fname}: '{old}' -> '{new}'")
    else:
        print(f"{fname}: '{old}' not found (already updated or different deck) — skipped")

In [ ]:
# --- 3.3 bump every sub-12pt text run up to 12pt (CBA1C03 requirement) -------
pattern = re.compile(r'sz="(\d{3,4})"')

def bump(m):
    return 'sz="1200"' if int(m.group(1)) < 1200 else m.group(0)

total_bumped = 0
for fpath in glob.glob(os.path.join(slides_dir, "slide*.xml")):
    txt = open(fpath, encoding="utf-8").read()
    before_small = len([x for x in pattern.findall(txt) if int(x) < 1200])
    if before_small:
        txt = pattern.sub(bump, txt)
        open(fpath, "w", encoding="utf-8").write(txt)
        total_bumped += before_small

print(f"Bumped {total_bumped} sub-12pt text runs to 12pt across the deck.")

In [ ]:
# --- 3.4 repackage the edited deck -------------------------------------------
EDITED_PPTX = "NM_NoShow_edited.pptx"
if os.path.exists(EDITED_PPTX):
    os.remove(EDITED_PPTX)

with zipfile.ZipFile(EDITED_PPTX, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files_ in os.walk(WORKDIR):
        for file_ in files_:
            full = os.path.join(root, file_)
            arcname = os.path.relpath(full, WORKDIR)
            zf.write(full, arcname)

# sanity check it opens
_check = Presentation(EDITED_PPTX)
print("Repackaged OK —", len(_check.slides._sldIdLst), "slides")

### Add the required Declaration of Originality and Generative AI Declaration slides

Both are inserted as front matter right after the cover slide — per the CBA1C03
brief, cover page / table of contents / references don't count toward the
18-slide content limit, and these declarations are the same kind of
non-content front matter (Annex B bundles them with the cover page template).


In [ ]:
NAVY = RGBColor(0x0B, 0x3D, 0x59)
DARK = RGBColor(0x1A, 0x20, 0x27)
TEAL = RGBColor(0x1C, 0x72, 0x93)
GREY = RGBColor(0x5B, 0x6B, 0x75)
GOLD = RGBColor(0xF4, 0xA3, 0x00)
LIGHT_BG = RGBColor(0xF7, 0xF9, 0xFA)
WHITE = RGBColor(0xFF, 0xFF, 0xFF)
BORDER = RGBColor(0xD8, 0xDE, 0xE6)

prs = Presentation(EDITED_PPTX)
SW, SH = prs.slide_width, prs.slide_height
blank_layout = prs.slide_masters[0].slide_layouts[0]

def header_bar(slide, eyebrow, title):
    bar = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, 0, 0, SW, Inches(1.05))
    bar.fill.solid(); bar.fill.fore_color.rgb = NAVY
    bar.line.fill.background(); bar.shadow.inherit = False
    tb = slide.shapes.add_textbox(Inches(0.55), Inches(0.12), Inches(11.5), Inches(0.8))
    tf = tb.text_frame; tf.word_wrap = True
    r0 = tf.paragraphs[0].add_run(); r0.text = eyebrow
    r0.font.size = Pt(12); r0.font.bold = True; r0.font.color.rgb = GOLD; r0.font.name = "Calibri"
    p1 = tf.add_paragraph(); r1 = p1.add_run(); r1.text = title
    r1.font.size = Pt(24); r1.font.bold = True; r1.font.color.rgb = WHITE; r1.font.name = "Cambria"

def add_note_box(slide, left, top, width, height, title_text, body_lines, title_color=TEAL):
    box = slide.shapes.add_shape(MSO_SHAPE.ROUNDED_RECTANGLE, left, top, width, height)
    box.adjustments[0] = 0.04
    box.fill.solid(); box.fill.fore_color.rgb = LIGHT_BG
    box.line.color.rgb = BORDER; box.line.width = Pt(0.75); box.shadow.inherit = False
    tf = box.text_frame; tf.word_wrap = True
    tf.margin_left = tf.margin_right = Inches(0.22)
    tf.margin_top = tf.margin_bottom = Inches(0.16)
    r0 = tf.paragraphs[0].add_run(); r0.text = title_text
    r0.font.bold = True; r0.font.size = Pt(13); r0.font.color.rgb = title_color; r0.font.name = "Calibri"
    for line in body_lines:
        p = tf.add_paragraph(); p.space_before = Pt(4)
        r = p.add_run(); r.text = line
        r.font.size = Pt(12); r.font.color.rgb = DARK; r.font.name = "Calibri"
    return box

print("Slide-building helpers ready.")

In [ ]:
# ---- Declaration of Originality slide ----
s1 = prs.slides.add_slide(blank_layout)
header_bar(s1, "FRONT MATTER  ·  NOT A CONTENT SLIDE", "Declaration of Originality")

body = s1.shapes.add_textbox(Inches(0.55), Inches(1.35), Inches(11.3), Inches(1.7))
tf = body.text_frame; tf.word_wrap = True
decl_lines = [
    "I am the originator of this work and I have appropriately acknowledged all other original sources "
    "used as my reference for this work.",
    "I understand that Plagiarism is the act of taking and using the whole or any part of another "
    "person's work, including work generated by AI, and presenting it as my own.",
    "I understand that Plagiarism is an academic offence, and if I am found to have committed or abetted "
    "the offence of plagiarism in relation to this submitted work, disciplinary action will be enforced.",
]
for i, line in enumerate(decl_lines):
    p = tf.paragraphs[0] if i == 0 else tf.add_paragraph()
    p.space_after = Pt(10)
    r = p.add_run(); r.text = line
    r.font.size = Pt(13); r.font.color.rgb = DARK; r.font.name = "Calibri"

table_shape = s1.shapes.add_table(2, 2, Inches(0.55), Inches(3.35), Inches(11.3), Inches(1.5))
table = table_shape.table
table.columns[0].width = Inches(6.65); table.columns[1].width = Inches(4.65)
for c, htext in enumerate(["Name of Student", "Signature"]):
    cell = table.cell(0, c)
    cell.fill.solid(); cell.fill.fore_color.rgb = NAVY
    cell.text_frame.paragraphs[0].text = htext
    run = cell.text_frame.paragraphs[0].runs[0]
    run.font.bold = True; run.font.size = Pt(13); run.font.color.rgb = WHITE; run.font.name = "Calibri"
for c in range(2):
    cell = table.cell(1, c)
    cell.fill.solid(); cell.fill.fore_color.rgb = WHITE
    cell.text_frame.paragraphs[0].text = "[To be completed before submission]"
    run = cell.text_frame.paragraphs[0].runs[0]
    run.font.size = Pt(12); run.font.italic = True; run.font.color.rgb = GREY; run.font.name = "Calibri"

add_note_box(s1, Inches(0.55), Inches(5.1), Inches(11.3), Inches(1.35), "Admin note", [
    "Complete the name, admin number, and signature fields above before submitting to PoliteMall.",
    "This declaration follows the Annex B template in the CBA1C03 project brief and is treated as front "
    "matter — it is not one of the graded content slides.",
])
print("Declaration of Originality slide added.")

In [ ]:
# ---- Generative AI Declaration slide ----
s2 = prs.slides.add_slide(blank_layout)
header_bar(s2, "FRONT MATTER  ·  NOT A CONTENT SLIDE", "Declaration on the Use of Generative AI Tools")

table_shape = s2.shapes.add_table(2, 1, Inches(0.55), Inches(1.3), Inches(11.3), Inches(1.1))
table = table_shape.table
hcell = table.cell(0, 0)
hcell.fill.solid(); hcell.fill.fore_color.rgb = NAVY
hcell.text_frame.paragraphs[0].text = "Links to Generative AI conversations (add before submission)"
r = hcell.text_frame.paragraphs[0].runs[0]
r.font.bold = True; r.font.size = Pt(13); r.font.color.rgb = WHITE; r.font.name = "Calibri"
bcell = table.cell(1, 0)
bcell.fill.solid(); bcell.fill.fore_color.rgb = WHITE
bcell.text_frame.paragraphs[0].text = "[Paste shareable conversation links or attach screenshots here]"
r = bcell.text_frame.paragraphs[0].runs[0]
r.font.size = Pt(12); r.font.italic = True; r.font.color.rgb = GREY; r.font.name = "Calibri"

add_note_box(s2, Inches(0.55), Inches(2.55), Inches(5.5), Inches(3.85), "How Generative AI was used in this project", [
    "1a. Introduction & Context — used to produce a first-draft business problem, persona, and JTBD "
    "framing; critiqued for gaps then refined using module concepts and domain knowledge.",
    "2a. Insights & Justifications — used together with SHAP output to help draft initial insight "
    "wording; checked against actual SHAP values and error analysis, then rewritten for accuracy.",
    "All AI output was verified against the underlying data/model results before inclusion.",
])
add_note_box(s2, Inches(6.25), Inches(2.55), Inches(5.6), Inches(3.85), "Citing non-recoverable AI output (APA 7th ed.)", [
    "AI conversations are non-recoverable sources: cite in-text as personal communication; do not add "
    "to the reference list.",
    "Format: (Communicator, personal communication, Month Day, Year)",
    "Example: (Paraphrase from OpenAI's ChatGPT AI language model, personal communication, March 8, 2023).",
    "Reminders: never copy AI output in totality; always verify claims; use clear, specific prompts.",
], title_color=GOLD)
print("Generative AI Declaration slide added.")

In [ ]:
# ---- reorder: Cover -> Declaration of Originality -> GenAI Declaration -> Agenda -> ... ----
xml_slides = prs.slides._sldIdLst
slides = list(xml_slides)
cover, agenda = slides[0], slides[1]
rest = slides[2:-2]
decl_orig, decl_ai = slides[-2], slides[-1]

new_order = [cover, decl_orig, decl_ai, agenda] + rest
for sld in list(xml_slides):
    xml_slides.remove(sld)
for sld in new_order:
    xml_slides.append(sld)

prs.save("NM_NoShow_Presentation.pptx")
print("Saved NM_NoShow_Presentation.pptx —", len(prs.slides._sldIdLst), "physical slides")

### Final checks

Quick structural checks — widescreen size, minimum font size, and slide order —
before you download the deck.


In [ ]:
from pptx import Presentation as _P
final = _P("NM_NoShow_Presentation.pptx")
print("Slide size:", final.slide_width, "x", final.slide_height,
      "(16:9 widescreen)" if final.slide_width == 12192000 else "CHECK THIS — not standard widescreen")
print("Total physical slides:", len(final.slides._sldIdLst))

for i, slide in enumerate(final.slides, 1):
    first_text = next((sh.text_frame.text.strip().splitlines()[0]
                        for sh in slide.shapes if sh.has_text_frame and sh.text_frame.text.strip()), "(no text)")
    print(i, "|", first_text[:70])

In [ ]:
from google.colab import files
files.download("NM_NoShow_Project_Report.docx")
files.download("NM_NoShow_Presentation.pptx")